# FarmaYA: Modelo de Machine Learning para la Predicción de Demanda de Medicamentos
### Clasificación Supervisada de Demanda Mensual (ALTA, MEDIA, BAJA) y Gestión de Inventarios

---
**Integrantes / Autores:** FarmaYA Development Team  
**Algoritmo Principal:** Random Forest Classifier  
**Aplicación:** Optimización de aprovisionamiento, cálculo de stock de seguridad dinámico y prevención de desabastecimiento / pérdidas por vencimiento.
---

## 1. Importación de Librerías
Importamos las librerías científicas y de aprendizaje automático requeridas: `pandas`, `numpy`, `scikit-learn` y herramientas de visualización.

In [ ]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Machine Learning con Scikit-Learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

print(" Librerías importadas correctamente.")

## 2. Generación y Carga del Dataset Histórico de Pedidos
Simulamos o extraemos el historial de pedidos de FarmaYA a lo largo de 12 meses para 12 medicamentos representativos con distintas tasas de rotación y requerimientos de receta médica.

In [ ]:
medicines_catalog = [
    # Alta rotación
    ("Ibuprofeno 800 mg", 120, 280, 1, 4, False),
    ("Paracetamol 500 mg", 150, 320, 1, 4, False),
    ("Ibuprofeno 400 mg", 90, 210, 1, 3, False),
    ("Vitamina C 1000 mg", 80, 190, 1, 3, False),
    # Media rotación
    ("Omeprazol 20 mg", 35, 75, 1, 2, False),
    ("Loratadina 10 mg", 30, 70, 1, 2, False),
    ("Suero oral Electrolit Fresa", 25, 60, 1, 3, False),
    # Baja rotación / Especializados con prescripción médica
    ("Amoxicilina 500 mg", 10, 25, 1, 2, True),
    ("Protector solar FPS 50+ Facial", 5, 18, 1, 1, False),
    ("Losartán 50 mg", 8, 22, 1, 2, True),
    ("Metformina 850 mg", 8, 20, 1, 2, True),
    ("Salbutamol Inhalador 100 mcg", 3, 14, 1, 1, True)
]

np.random.seed(42)
records = []
now = datetime.now()

for month_offset in range(12, 0, -1):
    base_date = now - timedelta(days=month_offset * 30)
    m_num = base_date.month
    for name, min_ord, max_ord, min_q, max_q, rx in medicines_catalog:
        seasonal_boost = 1.2 if m_num in [6, 7, 11, 12] and "Ibuprofeno" in name else 1.0
        num_orders = int(np.random.randint(min_ord, max_ord + 1) * seasonal_boost)
        for _ in range(num_orders):
            records.append({
                "created_at": base_date.replace(day=np.random.randint(1, 28)),
                "medicine": name,
                "quantity": np.random.randint(min_q, max_q + 1),
                "prescription": rx
            })

df_raw = pd.DataFrame(records)
print(f"Total de transacciones de pedidos generadas: {len(df_raw):,}")
df_raw.head(5)

## 3. Ingeniería de Características (Feature Engineering)
Agrupamos los datos a nivel mensual por medicamento y construimos las variables predictoras:
1. **Rezagos de Pedidos (`orders_lag_1`, `orders_lag_2`, `orders_lag_3`)**: Comportamiento en los meses $T-1$, $T-2$ y $T-3$.
2. **Media Móvil Trimestral (`rolling_avg_orders`)**: Tendencia central suavizada.
3. **Tendencia (`trend_orders`)**: Variación intermensual ($T-1 - T-2$).
4. **Rezagos de Unidades (`units_lag_1`, `units_lag_2`, `units_lag_3`)**: Volumen físico despachado.
5. **Ratio de Prescripción (`prescription_ratio`)** y **Mes calendario (`month_of_year`)**.

In [ ]:
df_raw["year_month"] = df_raw["created_at"].dt.to_period("M")

monthly = df_raw.groupby(["medicine", "year_month"]).agg(
    total_orders=("quantity", "count"),
    total_units=("quantity", "sum"),
    avg_units_per_order=("quantity", "mean"),
    prescription_ratio=("prescription", "mean")
).reset_index().sort_values(["medicine", "year_month"]).reset_index(drop=True)

# Umbrales empíricos de clasificación mediante terciles
q_low = int(monthly["total_orders"].quantile(0.33))
q_high = int(monthly["total_orders"].quantile(0.66))

def clasificar_demanda(pedidos):
    if pedidos <= q_low:
        return "BAJA"
    elif pedidos <= q_high:
        return "MEDIA"
    else:
        return "ALTA"

monthly["demand_category"] = monthly["total_orders"].apply(clasificar_demanda)
mapping = {"BAJA": 0, "MEDIA": 1, "ALTA": 2}
reverse_mapping = {0: "BAJA", 1: "MEDIA", 2: "ALTA"}
monthly["target"] = monthly["demand_category"].map(mapping)

print(f"Umbrales calculados: BAJA <= {q_low} | MEDIA: {q_low+1} a {q_high} | ALTA > {q_high}")
print("\nDistribución de clases históricas:")
print(monthly["demand_category"].value_counts())

In [ ]:
# Construcción de matriz de características con rezagos (Lags)
rows = []
for med, grp in monthly.groupby("medicine"):
    grp = grp.sort_values("year_month").reset_index(drop=True)
    for i in range(len(grp)):
        l1_o = grp.loc[i-1, "total_orders"] if i >= 1 else grp.loc[i, "total_orders"]
        l2_o = grp.loc[i-2, "total_orders"] if i >= 2 else l1_o
        l3_o = grp.loc[i-3, "total_orders"] if i >= 3 else l2_o
        
        l1_u = grp.loc[i-1, "total_units"] if i >= 1 else grp.loc[i, "total_units"]
        l2_u = grp.loc[i-2, "total_units"] if i >= 2 else l1_u
        l3_u = grp.loc[i-3, "total_units"] if i >= 3 else l2_u
        
        rolling_avg = (l1_o + l2_o + l3_o) / 3.0
        trend = l1_o - l2_o
        m_val = grp.loc[i, "year_month"].month
        
        rows.append({
            "medicine": med,
            "orders_lag_1": l1_o,
            "orders_lag_2": l2_o,
            "orders_lag_3": l3_o,
            "units_lag_1": l1_u,
            "units_lag_2": l2_u,
            "units_lag_3": l3_u,
            "rolling_avg_orders": rolling_avg,
            "trend_orders": trend,
            "avg_units_per_order": grp.loc[i, "avg_units_per_order"],
            "prescription_ratio": grp.loc[i, "prescription_ratio"],
            "month_of_year": m_val,
            "demand_category": grp.loc[i, "demand_category"],
            "target": grp.loc[i, "target"]
        })

dataset = pd.DataFrame(rows)
print(f"Matriz final de entrenamiento lista: {dataset.shape[0]} registros x {dataset.shape[1]} columnas.")
dataset.head(3)

## 4. Entrenamiento y Validación del Modelo (Random Forest)
Dividimos los datos en conjunto de entrenamiento (75%) y prueba (25%) mediante partición estratificada. Ajustamos un ensamble `RandomForestClassifier`.

In [ ]:
feature_cols = [
    "orders_lag_1", "orders_lag_2", "orders_lag_3",
    "units_lag_1", "units_lag_2", "units_lag_3",
    "rolling_avg_orders", "trend_orders",
    "avg_units_per_order", "prescription_ratio", "month_of_year"
]

X = dataset[feature_cols]
y = dataset["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42,
    class_weight="balanced"
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f" Exactitud Global del Modelo (Accuracy): {acc * 100:.2f}%")

## 5. Evaluación de Rendimiento: Matriz de Confusión y Reporte de Clasificación

In [ ]:
target_names = ["BAJA", "MEDIA", "ALTA"]
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, target_names=target_names))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[f"Real {t}" for t in target_names], columns=[f"Pred {t}" for t in target_names])
print("=== MATRIZ DE CONFUSIÓN ===")
print(cm_df)

## 6. Importancia de Variables (Feature Importance)
Permite explicar a la profesora qué factores estadísticos son los más influyentes al clasificar la demanda de un fármaco.

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("=== RANKING DE IMPORTANCIA DE CARACTERÍSTICAS ===")
for feat, imp in importances.items():
    bar = "█" * int(imp * 40)
    print(f"{feat:<22} | {bar:<30} {imp * 100:.1f}%")

## 7. Inferencia para el Próximo Mes y Stock de Seguridad Dinámico
Aplicamos el modelo a los datos más recientes para generar la proyección del próximo mes junto con la política de abastecimiento:
- **ALTA:** Margen de seguridad $+35\%$
- **MEDIA:** Margen de seguridad $+20\%$
- **BAJA:** Margen de seguridad $+10\%$

In [ ]:
predictions = []
next_month = (now + timedelta(days=30)).month

for med, grp in monthly.groupby("medicine"):
    last = grp.iloc[-1]
    prev1 = grp.iloc[-2] if len(grp) >= 2 else last
    prev2 = grp.iloc[-3] if len(grp) >= 3 else prev1
    
    l1_o = last["total_orders"]
    l2_o = prev1["total_orders"]
    l3_o = prev2["total_orders"]
    
    rolling_avg = (l1_o + l2_o + l3_o) / 3.0
    trend = l1_o - l2_o
    est_orders = max(1, int(round(rolling_avg + (trend * 0.5))))
    est_units = int(round(est_orders * last["avg_units_per_order"]))
    
    feat_vector = [
        l1_o, l2_o, l3_o,
        last["total_units"], prev1["total_units"], prev2["total_units"],
        rolling_avg, trend,
        last["avg_units_per_order"], last["prescription_ratio"], next_month
    ]
    
    scaled_v = scaler.transform([feat_vector])
    pred_cat = reverse_mapping[model.predict(scaled_v)[0]]
    conf = np.max(model.predict_proba(scaled_v)[0]) * 100.0
    
    # Regla de Stock de Seguridad
    if pred_cat == "ALTA":
        stock_rec = int(round(est_units * 1.35))
        accion = "Aprovisionamiento masivo prioritario (+35%)"
    elif pred_cat == "MEDIA":
        stock_rec = int(round(est_units * 1.20))
        accion = "Reposición regular estándar (+20%)"
    else:
        stock_rec = int(round(est_units * 1.10))
        accion = "Compra bajo demanda (+10%) para evitar caducidad"
        
    predictions.append({
        "Medicamento": med,
        "Demanda Proyectada": pred_cat,
        "Confianza (%)": round(conf, 1),
        "Pedidos Estimados": est_orders,
        "Unidades Proyectadas": est_units,
        "Stock Sugerido (Bodega)": stock_rec,
        "Estrategia Operativa": accion
    })

df_res = pd.DataFrame(predictions).sort_values(by="Pedidos Estimados", ascending=False)
df_res

## 8. Conclusiones y Valor Estratégico
- El modelo Random Forest logra anticipar con alta precisión las variaciones de demanda intermensuales.
- La combinación de **rezagos de pedidos**, **estacionalidad** y **tendencia** proporciona una sólida capacidad de generalización.
- La integración de estas predicciones con reglas de **Stock de Seguridad dinámico** protege la continuidad de la atención farmacéutica sin incurrir en costos por sobreinventario ni vencimiento de medicamentos.